# Wedding planner — an Algerian wedding 💍

**DJEF Wedding** — a small Module 2 project: one LangChain agent that plans a real
Algerian **عرس** (wedding). Four tools, local data only — no web search needed:

| tool | answers |
|---|---|
| `wedding_checklist` | what to do, when — 3 months before → the day after (hennoutia, l'aâchira, zaffa, nişar) |
| `estimate_budget` | budget in **Algerian dinars (DA)** by guest count and style |
| `wilaya_traditions` | dress, music, sweets and signature ritual by wilaya |
| `menu_per_guests` | traiteur menu and quantities (djaj b'hazzouan, dolma, baklawa…) |

The notebook is the bench; `streamlit/app.py` is the small app, built on the same
service module (`wedding_planner_service.py`). Only a Gemini key is needed — `.env`
at the project root: `GOOGLE_API_KEY`. Same small, cheap model as every notebook in
this path: `gemini-3.1-flash-lite`.

> Budget figures and portions are **indicative examples** — adjust them to the real
> price list before the big day.

## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

def _check_install(pkg: str, mod: str):
    if _ilu.find_spec(mod) is None:
        result = _sp.run([_sys.executable, "-m", "pip", "install", "-q", pkg],
                         capture_output=True, text=True)
        print(f"{pkg}: installed" if result.returncode == 0
              else f"{pkg}: install FAILED -> {result.stderr[-200:]}")
    else:
        print(f"{pkg}: already installed")

for _pkg, _mod in [
    ("python-dotenv", "dotenv"),
    ("langchain", "langchain"),
    ("langgraph", "langgraph"),
    ("google-genai", "google.genai"),
]:
    _check_install(_pkg, _mod)

print("Setup done.")

python-dotenv: already installed
langchain: already installed
langgraph: already installed
google-genai: already installed
Setup done.


In [2]:
# Load the Gemini key from the nearest .env (project folder) — never inside the file.
import os

import wedding_planner_service as wp

source = wp.load_keys()
print("Key source:", source)
print("GOOGLE_API_KEY:", "set" if os.getenv("GOOGLE_API_KEY") else "missing")
print("GEMINI_API_KEY:", "set" if os.getenv("GEMINI_API_KEY") else "missing")
print("Setup done.")

ModuleNotFoundError: No module named 'wedding_planner_service'

In [ ]:
# Same small, cheap Gemini model as every notebook in this path.
MODEL = wp.WEDDING_MODEL
print("Model:", MODEL)

## 1. The checklist — what happens, when

The data lives **in the tool**, not in the prompt: stages from *3 months before* to
*the day after*, with the Algerian signatures (hennoutia, l'aâchira, zaffa, nişar).

In [ ]:
import json

stages = json.loads(wp.wedding_checklist.invoke({}))
for stage in stages["stages"]:
    print(f"{stage['stage']}:")
    for task in stage["tasks"]:
        print("   -", task)
    print()

## 2. The budget — in Algerian dinars (DA)

One tool, a guest count and a style (`simple`, `classic`, `luxury`): fixed costs
(salon des fêtes, hennoutia, orfi, photo, décoration, daftari, tenue, nişar) plus
per-guest costs (traiteur + sweets).

In [ ]:
budget = json.loads(wp.estimate_budget.invoke({"guests": 100, "style": "classic"}))
for item, price in budget["breakdown_da"].items():
    print(f"  {item:44} {price:>9} DA")
print(f"  {'TOTAL':44} {budget['total_da']:>9} DA")
print("  per guest:", budget["per_guest_da"], "DA |", budget["note"])

## 3. Traditions by wilaya

Dress, music, sweets and the signature ritual — data for six wilayas, with a friendly
fallback that lists the known ones.

In [ ]:
trad = json.loads(wp.wilaya_traditions.invoke({"wilaya": "Tlemcen"}))
t = trad["traditions"]
print(f"Wilaya: {trad['wilaya'].title()}")
for key in ("dress", "music", "sweets", "signature"):
    print(f"  {key:10}: {t[key]}")

## 4. The traiteur — menu per guests

Portions per guest and totals for the whole wedding — the tool does the arithmetic,
the agent only reports it.

In [ ]:
menu = json.loads(wp.menu_per_guests.invoke({"guests": 100}))
print(f"Menu for {menu['guests']} guests — per guest  ->  total")
for name, per in menu["per_guest"].items():
    print(f"  {name:22} {per:14} -> {menu['totals'][name]}")
print(menu["note"])

## 5. The agent — one planner, four tools

Same factory as everywhere in the path: `create_agent` with a short system prompt.
The prompt says *which tool for which question* and *never invent the numbers*.

In [ ]:
agent = wp.build_wedding_agent(model=MODEL)
tool_names = [t.name for t in
              (wp.wedding_checklist, wp.estimate_budget, wp.wilaya_traditions, wp.menu_per_guests)]
print("DJEF Wedding ready — tools:", tool_names)

## 6. Test — ask for a plan

One question, four tools: the agent should call the checklist, the budget, the
traditions **and** the menu, then compose one plan.

In [ ]:
from langchain.messages import HumanMessage

question = (
    "We are 100 guests for a wedding in Tlemcen on 20 June 2027, classic style. "
    "Give me the plan: timeline, budget in DA, local traditions and the menu."
)
response = agent.invoke({"messages": [HumanMessage(content=question)]})
print(wp.last_text(response))

## 7. Trace — the same run, step by step

The printout above shows only the final plan. The trace shows the *decisions*: which
tool the planner picked, with which arguments, node by node.

In [ ]:
for update in agent.stream({"messages": [HumanMessage(content=question)]},
                           stream_mode="updates"):
    for node, payload in update.items():
        for msg in payload.get("messages", []):
            kind = type(msg).__name__.replace("Message", "")
            if getattr(msg, "tool_calls", None):
                call = msg.tool_calls[0]
                detail = f"-> {call['name']}({call['args']})"
            elif kind == "Tool":
                detail = f"{str(msg.content)[:70]}..."
            elif isinstance(msg.content, str):
                detail = msg.content[:70] + ("..." if len(msg.content) > 70 else "")
            elif msg.content:
                detail = msg.content[0].get("text", "")[:70] + "..."
            else:
                detail = ""
            print(f"[{node}] {kind}: {detail}")

## 8. The Streamlit app

The app imports the same `wedding_planner_service` — one form, one button:

```bash
# from the project folder (project_wedding_planner/):
streamlit run streamlit/app.py
```

Sidebar: guests, wilaya, style, date. Type a request → the agent answers with the plan.

## Summary of functions

| Function / class | Simple definition |
|---|---|
| `load_keys()` | Reads the Gemini key from the nearest `.env` (project or notebooks folder). |
| `@tool wedding_checklist()` | Static data behind a tool: the staged Algerian wedding timeline. |
| `@tool estimate_budget(guests, style)` | Budget breakdown in DA for a guest count and a style. |
| `@tool wilaya_traditions(wilaya)` | Dress, music, sweets and ritual for one wilaya. |
| `@tool menu_per_guests(guests)` | Traiteur menu + total quantities for N guests. |
| `create_agent(model, tools, system_prompt)` | The agent factory — one planner, four tools. |
| `agent.stream(..., stream_mode="updates")` | The trace: every node of the same run. |
| `last_text(response)` | Last message as plain text (Gemini content may be blocks). |

By
> Kassouar Fatima                   26-09-2026